# 05 — Token Budgeting and Allocation — Practical Examples

**Covers**: ContextBudget dataclass, TokenBudgetEnforcer, dynamic reallocation, prioritized tools, budget dashboard

In [ ]:
import os, json
import tiktoken
from openai import OpenAI
from dataclasses import dataclass
from copy import copy
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
enc = tiktoken.encoding_for_model(MODEL)
print('✅ Setup complete')

## 1. ContextBudget — Define the Budget

In [ ]:
@dataclass
class ContextBudget:
    context_limit:   int
    system_budget:   int
    tools_budget:    int
    history_budget:  int
    current_budget:  int
    output_budget:   int

    @property
    def total_input_budget(self):
        return self.system_budget + self.tools_budget + self.history_budget + self.current_budget

    @property
    def safety_margin(self):
        return self.context_limit - self.total_input_budget - self.output_budget

    @property
    def is_valid(self):
        return self.safety_margin >= 0

    def report(self):
        pct = self.total_input_budget / self.context_limit * 100
        print(f"{'Budget Section':<20} {'Tokens':>8} {'% of Total':>12}")
        print('-' * 45)
        for name, val in [('System', self.system_budget), ('Tools', self.tools_budget),
                          ('History', self.history_budget), ('Current msg', self.current_budget),
                          ('Output reserve', self.output_budget)]:
            print(f"{name:<20} {val:>8,} {val/self.context_limit*100:>11.1f}%")
        print('-' * 45)
        print(f"{'Total Input':<20} {self.total_input_budget:>8,} {pct:>11.1f}%")
        print(f"{'Safety margin':<20} {self.safety_margin:>8,}")

BUDGET = ContextBudget(
    context_limit  = 128_000,
    system_budget  =   2_000,
    tools_budget   =   2_000,
    history_budget =  50_000,
    current_budget =   5_000,
    output_budget  =   4_000,
)
BUDGET.report()

## 2. TokenBudgetEnforcer — Enforce Each Section

In [ ]:
class TokenBudgetEnforcer:
    def __init__(self, budget: ContextBudget, model: str = MODEL):
        self.budget = budget
        self.enc = tiktoken.encoding_for_model(model)

    def _count(self, text: str) -> int:
        return len(self.enc.encode(text))

    def _count_msgs(self, msgs: list[dict]) -> int:
        return sum(3 + self._count(str(m.get('content',''))) for m in msgs) + 3

    def enforce_system(self, system: str) -> str:
        t = self._count(system)
        if t <= self.budget.system_budget: return system
        ids = self.enc.encode(system)[:self.budget.system_budget - 30]
        print(f"  ⚠️  System: {t} → ~{self.budget.system_budget} tokens (truncated)")
        return self.enc.decode(ids) + '\n[TRUNCATED]'

    def enforce_history(self, history: list[dict]) -> tuple[list[dict], int]:
        total = self._count_msgs(history)
        if total <= self.budget.history_budget: return history, total
        kept, used, dropped = [], 3, 0
        for m in reversed(history):
            t = 3 + self._count(str(m.get('content','')))
            if used + t <= self.budget.history_budget: kept.append(m); used += t
            else: dropped += 1
        kept.reverse()
        print(f"  ⚠️  History: dropped {dropped} oldest messages")
        return kept, used

    def enforce_tools(self, tools: list[dict]) -> list[dict]:
        kept, used = [], 15
        for tool in tools:
            t = self._count(json.dumps(tool))
            if used + t <= self.budget.tools_budget: kept.append(tool); used += t
            else: print(f"  ⚠️  Tool '{tool['function']['name']}' removed (budget full)")
        return kept

    def usage_report(self, messages: list[dict], tools: list[dict] = None) -> dict:
        mt = self._count_msgs(messages)
        tt = self._count(json.dumps(tools)) + 15 if tools else 0
        total = mt + tt + self.budget.output_budget
        return {'message_tokens': mt, 'tool_tokens': tt,
                'total_used': total, 'headroom': self.budget.context_limit - total,
                'pct': round(total / self.budget.context_limit * 100, 1)}

# Test with oversized history
enforcer = TokenBudgetEnforcer(ContextBudget(
    context_limit=128_000, system_budget=200, tools_budget=500,
    history_budget=400, current_budget=200, output_budget=500
))

big_history = [
    {'role': 'user' if i%2==0 else 'assistant',
     'content': f'Turn {i//2+1} - this is a {"question" if i%2==0 else "detailed answer"} about Python topic {i//2+1} with lots of information.'}
    for i in range(20)
]

trimmed, _ = enforcer.enforce_history(big_history)
print(f"Original: {len(big_history)} messages → Trimmed: {len(trimmed)} messages")

## 3. Dynamic Budget Reallocation

In [ ]:
def reallocate(budget: ContextBudget, extra_system: int = 0, extra_history: int = 0) -> ContextBudget:
    b = copy(budget)
    if extra_system > 0:
        borrow = min(extra_system, b.history_budget // 4)
        b.system_budget  += borrow
        b.history_budget -= borrow
        print(f"  Borrowed {borrow} tokens from history → system")
    if extra_history > 0:
        borrow = min(extra_history, b.output_budget // 2)
        b.history_budget += borrow
        b.output_budget  -= borrow
        print(f"  Borrowed {borrow} tokens from output → history")
    return b if b.is_valid else budget

print("Original budget:")
print(f"  system={BUDGET.system_budget}, history={BUDGET.history_budget}, output={BUDGET.output_budget}")
new_budget = reallocate(BUDGET, extra_system=500, extra_history=2000)
print("After reallocation:")
print(f"  system={new_budget.system_budget}, history={new_budget.history_budget}, output={new_budget.output_budget}")

## 4. Budget Dashboard — Visual Monitoring

In [ ]:
def budget_dashboard(report: dict, limit: int):
    pct = report['pct']
    bar = '█' * int(pct / 100 * 40) + '░' * (40 - int(pct / 100 * 40))
    icon = '🟢' if pct < 60 else ('🟡' if pct < 85 else '🔴')
    print(f"\n{'─'*52}")
    print(f"  Context Budget {icon} {pct:.1f}% used")
    print(f"  [{bar}]")
    print(f"  {'Messages:':<15} {report['message_tokens']:>7,} tokens")
    print(f"  {'Tools:':<15} {report['tool_tokens']:>7,} tokens")
    print(f"  {'Output Rsv:':<15} {BUDGET.output_budget:>7,} tokens")
    print(f"  {'Total:':<15} {report['total_used']:>7,} / {limit:,}")
    print(f"  {'Headroom:':<15} {report['headroom']:>7,} tokens")
    print(f"{'─'*52}")

# Live demo with a real API call
enforcer2 = TokenBudgetEnforcer(BUDGET)
system = 'You are a helpful assistant.'
history = [{'role': 'user', 'content': 'Hello'}, {'role': 'assistant', 'content': 'Hi there!'}]
current = 'What is token budgeting?'

safe_sys = enforcer2.enforce_system(system)
safe_hist, _ = enforcer2.enforce_history(history)
messages = [{'role': 'system', 'content': safe_sys}, *safe_hist, {'role': 'user', 'content': current}]

report = enforcer2.usage_report(messages)
budget_dashboard(report, BUDGET.context_limit)

r = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=100)
print(f"\nAnswer: {r.choices[0].message.content[:150]}")

## 📌 Summary

- **5 budget sections**: system, tools, history, current, output — each needs an allocation
- **TokenBudgetEnforcer**: enforce each section, trim/drop if exceeded
- **Dynamic reallocation**: borrow from flexible sections (history/output) when needed
- **Dashboard**: visual token utilization, headroom, pct — monitor every call
- **Budget before running** — like allocating RAM before executing a program